In [ ]:
import sqlite3
import pandas as pd

conn = sqlite3.connect(':memory:')
cursor = conn.cursor()

cursor.execute('''
CREATE TABLE customers (
    customer_id INT PRIMARY KEY,
    name VARCHAR(50),
    city VARCHAR(50)
);
''')

cursor.execute('''
CREATE TABLE orders (
    order_id INT PRIMARY KEY,
    customer_id INT REFERENCES customers(customer_id),
    order_date DATE,
    amount NUMERIC(10, 2),
    status VARCHAR(20)
);
''')

customers_data = [
    (1, 'Aarav', 'Bengaluru'),
    (2, 'Priya', 'Mumbai'),
    (3, 'Rohan', 'Delhi'),
    (4, 'Meera', 'Pune'),
    (5, 'Kabir', 'Hyderabad')
]
cursor.executemany('INSERT INTO customers VALUES (?, ?, ?);', customers_data)

orders_data = [
    (101, 1, '2026-08-25', 2200, 'delivered'),
    (102, 1, '2026-09-10', 1800, 'delivered'),
    (103, 1, '2026-09-25', 2500, 'delivered'),
    (104, 2, '2026-09-01', 3000, 'delivered'),
    (105, 2, '2026-10-15', 2800, 'delivered'),
    (106, 2, '2026-11-24', 3200, 'delivered'),

    # Rohan: 1 delivered + 1 cancelled order (should be excluded)
    (107, 3, '2026-08-01', 6000, 'delivered'),
    (108, 3, '2026-08-15', 2000, 'cancelled'),

    # Meera: Low total spend under 5,000 (should be excluded)
    (109, 4, '2026-08-10', 800, 'delivered'),
    (110, 4, '2026-09-05', 900, 'delivered'),

    # Kabir: VIP who naturally orders every ~61 days, silent for 54 days (NOT OVERDUE)
    (111, 5, '2026-06-01', 2500, 'delivered'),
    (112, 5, '2026-08-01', 2600, 'delivered'),
    (113, 5, '2026-10-01', 2400, 'delivered')
]
cursor.executemany('INSERT INTO orders VALUES (?, ?, ?, ?, ?);', orders_data)
conn.commit()

print("Database initialized and tables populated successfully!\n")

# Correctly wrap the SQL in Python using pandas
query = '''
WITH delivered_orders AS (
    SELECT
        o.customer_id,
        c.name,
        c.city,
        o.order_date,
        o.amount,
        -- Anchor "today" to the latest date in the dataset
        MAX(o.order_date) OVER () AS as_of_date,
        -- Days elapsed since previous order (using julianday in SQLite)
        CAST(julianday(o.order_date) - julianday(LAG(o.order_date) OVER (
            PARTITION BY o.customer_id
            ORDER BY o.order_date
        )) AS INT) AS days_since_prev_order
    FROM orders o
    JOIN customers c ON o.customer_id = c.customer_id
    WHERE o.status = 'delivered'
),
customer_summary AS (
    SELECT
        customer_id,
        name,
        city,
        COUNT(*) AS total_orders,
        SUM(amount) AS total_spend,
        MAX(order_date) AS last_order_date,
        CAST(julianday(MAX(as_of_date)) - julianday(MAX(order_date)) AS INT) AS days_inactive,
        ROUND(AVG(days_since_prev_order), 1) AS avg_order_gap_days
    FROM delivered_orders
    GROUP BY customer_id, name, city
    HAVING COUNT(*) >= 2
       AND SUM(amount) >= 5000
)
SELECT
    customer_id,
    name,
    city,
    total_orders,
    total_spend,
    last_order_date,
    days_inactive,
    avg_order_gap_days,
    CASE
        WHEN days_inactive > avg_order_gap_days * 1.5 THEN 'High Churn Risk - Send Offer'
        ELSE 'Normal Buying Cycle - Do Not Discount'
    END AS action_status
FROM customer_summary
WHERE days_inactive >= 45
ORDER BY (CAST(days_inactive AS REAL) / COALESCE(NULLIF(avg_order_gap_days, 0), 1)) DESC;
'''

df = pd.read_sql_query(query, conn)
print(df)


Database initialized and tables populated successfully!

   customer_id   name       city  total_orders  total_spend last_order_date  \
0            1  Aarav  Bengaluru             3         6500      2026-09-25   
1            5  Kabir  Hyderabad             3         7500      2026-10-01   

   days_inactive  avg_order_gap_days                          action_status  
0             60                15.5           High Churn Risk - Send Offer  
1             54                61.0  Normal Buying Cycle - Do Not Discount  
